In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Path to cleaned datasets
DATA_PATH = "../outputs/cleaned_data/"

# Load cleaned datasets
competitions = pd.read_csv(DATA_PATH + "competitions_cleaned.csv")
teams = pd.read_csv(DATA_PATH + "teams_cleaned.csv")
matches = pd.read_csv(DATA_PATH + "matches_cleaned.csv")
ball_by_ball = pd.read_csv(DATA_PATH + "ball_by_ball_cleaned.csv")
player_mapping = pd.read_csv(DATA_PATH + "player_mapping_cleaned.csv")
playing_eleven = pd.read_csv(DATA_PATH + "playing_eleven_cleaned.csv")

# Convert match date
matches["match_date"] = pd.to_datetime(matches["match_date"])

# Display dataset information
datasets = {
    "competitions": competitions,
    "teams": teams,
    "matches": matches,
    "ball_by_ball": ball_by_ball,
    "player_mapping": player_mapping,
    "playing_eleven": playing_eleven
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

competitions: (141, 4)
teams: (353, 3)
matches: (883, 22)
ball_by_ball: (200999, 33)
player_mapping: (6413, 6)
playing_eleven: (28381, 9)


In [2]:
# COMPETITION AND SEASON ANALYSIS

print("TOTAL COMPETITIONS:", competitions["comp_id"].nunique())

print("\nMATCH FORMATS")
display(competitions["match_format"].value_counts())

print("\nSEASON DISTRIBUTION")
display(
    competitions.groupby("season")
    .agg(
        total_competitions=("comp_id", "nunique"),
        match_formats=("match_format", "nunique")
    )
    .sort_index()
)

print("\nCOMPETITION DETAILS")
display(competitions.head(10))

TOTAL COMPETITIONS: 141

MATCH FORMATS


match_format
T20I    134
T20       7
Name: count, dtype: int64


SEASON DISTRIBUTION


,total_competitions,match_formats
season,,
2022,25,2
2022/23,3,1
2023,20,2
2023/24,1,1
2024,33,2
2024/25,1,1
2025,33,2
2026,25,2



COMPETITION DETAILS


,comp_id,comp_name,match_format,season
0,3395,South Africa in New Zealand T20I Series 2026,T20I,2026
1,3405,Afghanistan v Sri Lanka T20I Series 2026,T20I,2026
2,3465,India in Zimbabwe T20I Series 2026,T20I,2026
3,3568,Bangladesh in Zimbabwe T20I Series 2026,T20I,2026
4,3482,Australia in Bangladesh T20I Series 2026,T20I,2026
5,3464,Sri Lanka in West Indies T20I Series 2026,T20I,2026
6,3765,Sri Lanka in Pakistan T20I Series 2026,T20I,2026
7,3739,India in Japan T20I Match 2026,T20I,2026
8,3787,Afghanistan v Zimbabwe T20I Series 2026,T20I,2026
9,3713,Afghanistan vs India in India T20I Series 2026,T20I,2026


In [3]:
# MATCH DISTRIBUTION BY SEASON

match_analysis = matches.copy()

# Connect matches with competition information
match_analysis = match_analysis.merge(
    competitions[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

print("TOTAL MATCHES:", match_analysis["match_id"].nunique())

print("\nMATCHES BY SEASON")
display(
    match_analysis.groupby("season")
    .agg(
        total_matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique")
    )
    .sort_index()
)

print("\nMATCHES BY FORMAT")
display(
    match_analysis.groupby("match_format")
    .agg(
        total_matches=("match_id", "nunique"),
        seasons=("season", "nunique")
    )
)

TOTAL MATCHES: 883

MATCHES BY SEASON


,total_matches,competitions
season,,
2022,168,25
2022/23,12,3
2023,137,20
2023/24,3,1
2024,198,33
2024/25,3,1
2025,216,32
2026,146,23



MATCHES BY FORMAT


,total_matches,seasons
match_format,,
T20,401,5
T20I,482,8


In [4]:
# MATCH DISTRIBUTION BY COMPETITION

competition_match_summary = (
    match_analysis.groupby(
        ["comp_name", "match_format", "season"],
        dropna=False
    )
    .agg(
        total_matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["match_format", "season", "total_matches"],
        ascending=[True, True, False]
    )
)

display(competition_match_summary.to_string(index=False))

'                                              comp_name match_format  season  total_matches\n                                               IPL 2022          T20    2022             74\n                                               IPL 2023          T20    2023             74\n                                               IPL 2024          T20    2024             74\n                  Mens T20 Emerging Teams Asia Cup 2024          T20    2024             15\n                                               IPL 2025          T20    2025             75\n                             Asia Cup Rising Stars 2025          T20    2025             15\n                                               IPL 2026          T20    2026             74\n                            Mens Twenty20 Asia Cup 2022         T20I    2022             13\n                   England in Pakistan T20I Series 2022         T20I    2022              7\n                           Asia Cup T20 Qualifiers 2022         T20I 

In [5]:
# IDENTIFY IPL MATCHES

ipl_matches = match_analysis[
    match_analysis["comp_name"].str.fullmatch(
        r"IPL \d{4}",
        na=False
    )
].copy()

print("IPL MATCHES:", ipl_matches["match_id"].nunique())

print("\nIPL MATCHES BY SEASON:")
display(
    ipl_matches.groupby("season")
    .agg(
        total_matches=("match_id", "nunique")
    )
    .sort_index()
)

print("\nOTHER T20 COMPETITIONS:")
display(
    match_analysis[
        (match_analysis["match_format"] == "T20") &
        (~match_analysis["comp_name"].str.fullmatch(r"IPL \d{4}", na=False))
    ][["comp_name", "season"]]
    .drop_duplicates()
    .sort_values(["season", "comp_name"])
)

IPL MATCHES: 371

IPL MATCHES BY SEASON:


,total_matches
season,
2022,74
2023,74
2024,74
2025,75
2026,74



OTHER T20 COMPETITIONS:


,comp_name,season
499,Mens T20 Emerging Teams Asia Cup 2024,2024
733,Asia Cup Rising Stars 2025,2025


In [6]:
# IPL MATCH RESULT SUMMARY

print("IPL MATCH RESULT STATUS:")
display(ipl_matches["result"].value_counts(dropna=False))

print("\nIPL WINNER INFORMATION:")
print("Missing winner IDs:", ipl_matches["win_team_id"].isna().sum())
print("Matches with winner IDs:", ipl_matches["win_team_id"].notna().sum())

print("\nIPL MATCHES BY RESULT TYPE:")
display(
    ipl_matches.groupby(["season", "result"], dropna=False)
    .size()
    .unstack(fill_value=0)
)

IPL MATCH RESULT STATUS:


result
Delhi Capitals Won by 6 wickets               7
Kolkata Knight Riders Won by 8 wickets        6
Punjab Kings Won by 5 wickets                 5
Match Abandoned                               5
Gujarat Titans Won by 7 wickets               5
                                             ..
Sunrisers Hyderabad Won by 55 runs            1
Rajasthan Royals Won by 30 runs               1
Delhi Capitals Won by 40 runs                 1
Royal Challengers Bengaluru Won by 92 runs    1
Rajasthan Royals Won by 47 runs               1
Name: count, Length: 283, dtype: int64


IPL WINNER INFORMATION:
Missing winner IDs: 9
Matches with winner IDs: 362

IPL MATCHES BY RESULT TYPE:


result,Chennai Super Kings Won by 103 runs,Chennai Super Kings Won by 12 runs,Chennai Super Kings Won by 15 runs,Chennai Super Kings Won by 2 wickets,Chennai Super Kings Won by 20 runs,Chennai Super Kings Won by 23 Runs,Chennai Super Kings Won by 23 runs,Chennai Super Kings Won by 27 Runs,Chennai Super Kings Won by 28 runs,Chennai Super Kings Won by 3 Wickets,...,Sunrisers Hyderabad Won by 65 runs,Sunrisers Hyderabad Won by 67 runs,Sunrisers Hyderabad Won by 8 Wickets,Sunrisers Hyderabad Won by 8 wickets,Sunrisers Hyderabad Won by 9 Runs,Sunrisers Hyderabad won by 31 runs,Sunrisers Hyderabad won by 7 wicket,Sunrisers Hyderabad won by 7 wickets,Sunrisers Hyderabad won by 8 wickets,Sunrisers Hyderabad won by 9 wickets
season,,,,,,,,,,,,,,,,,,,,,
2022,0,0,0,0,0,1,0,0,0,1,...,0,0,1,0,0,0,1,1,1,1
2023,0,1,1,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,1,0
2024,0,0,0,0,1,0,0,0,1,0,...,0,1,0,0,0,1,0,0,0,0
2025,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2026,1,0,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [7]:
# IPL MATCHES WITHOUT A RECORDED WINNER

no_winner_matches = ipl_matches[
    ipl_matches["win_team_id"].isna()
].copy()

columns_to_check = [
    "match_id",
    "season",
    "match_date",
    "comp_name",
    "team1_id",
    "team2_id",
    "result",
    "win_team_id",
    "win_margin",
    "win_outcome"
]

display(
    no_winner_matches[
        [col for col in columns_to_check if col in no_winner_matches.columns]
    ].sort_values("match_date")
)

,match_id,season,match_date,comp_name,result,win_team_id
234,20226,2023,2023-05-03,IPL 2023,Match Abandoned,NaN
396,31281,2024,2024-05-13,IPL 2024,Match Abandoned,NaN
399,31284,2024,2024-05-16,IPL 2024,Match Abandoned,NaN
403,31288,2024,2024-05-19,IPL 2024,Match Abandoned,NaN
569,40737,2025,2025-04-26,IPL 2025,No Result,NaN
580,40748,2025,2025-05-05,IPL 2025,No Result,NaN
583,40751,2025,2025-05-08,IPL 2025,No Result,NaN
593,40761,2025,2025-05-17,IPL 2025,Match Abandoned,NaN
759,50136,2026,2026-04-06,IPL 2026,No Result,NaN


In [8]:
# IPL DATE COVERAGE

print("FIRST IPL MATCH:", ipl_matches["match_date"].min())
print("LAST IPL MATCH:", ipl_matches["match_date"].max())

print("\nMATCHES BY YEAR:")
display(
    ipl_matches.groupby(ipl_matches["match_date"].dt.year)
    .agg(
        total_matches=("match_id", "nunique"),
        first_match=("match_date", "min"),
        last_match=("match_date", "max")
    )
)

FIRST IPL MATCH: 2022-03-26 00:00:00
LAST IPL MATCH: 2026-05-31 00:00:00

MATCHES BY YEAR:


,total_matches,first_match,last_match
match_date,,,
2022,74,2022-03-26,2022-05-29
2023,74,2023-03-31,2023-05-29
2024,74,2024-03-22,2024-05-26
2025,75,2025-03-22,2025-06-03
2026,74,2026-03-28,2026-05-31


In [9]:
# IPL INNINGS DISTRIBUTION

ipl_deliveries = ball_by_ball[
    ball_by_ball["match_id"].isin(ipl_matches["match_id"])
].copy()

print("TOTAL IPL DELIVERIES:", len(ipl_deliveries))
print("\nINNINGS DISTRIBUTION:")

display(
    ipl_deliveries.groupby("innings")
    .agg(
        total_deliveries=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .sort_index()
)

print("\nMATCHES WITH MORE THAN TWO INNINGS:")

innings_per_match = (
    ipl_deliveries.groupby("match_id")["innings"]
    .nunique()
)

display(innings_per_match[innings_per_match > 2])

TOTAL IPL DELIVERIES: 87687

INNINGS DISTRIBUTION:


,total_deliveries,matches
innings,,
1,45588,367
2,42085,363
3,9,2
4,5,2



MATCHES WITH MORE THAN TWO INNINGS:


match_id
40725    4
50359    4
Name: innings, dtype: int64

In [12]:
# INSPECT IPL MATCHES WITH SUPER-OVER INNINGS

super_over_match_ids = [40725, 50359]

display(
    ipl_deliveries[
        ipl_deliveries["match_id"].isin(super_over_match_ids)
    ].sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    )[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_id",
            "striker_name",
            "bowler_id",
            "bowler_name",
            "runs",
            "is_wicket"
        ]
    ].to_string(index=False)
)

' match_id  innings  overid  over_ball_num  striker_id         striker_name  bowler_id        bowler_name  runs  is_wicket\n    40725        1       1              1        7341          Jake McGurk       3298       Jofra Archer     0      False\n    40725        1       1              2        7341          Jake McGurk       3298       Jofra Archer     0      False\n    40725        1       1              3        7341          Jake McGurk       3298       Jofra Archer     4      False\n    40725        1       1              4        7341          Jake McGurk       3298       Jofra Archer     4      False\n    40725        1       1              5        7341          Jake McGurk       3298       Jofra Archer     1      False\n    40725        1       1              6       19714        Abishek Porel       3298       Jofra Archer     1      False\n    40725        1       2              1       19714        Abishek Porel       6806   Tushar Deshpande     4      False\n    40725      

In [11]:
# CHECK ACTUAL DELIVERY COLUMN NAMES

print(ipl_deliveries.columns.tolist())

['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']


In [13]:
# SEPARATE REGULAR INNINGS FROM SUPER OVERS

ipl_regular_deliveries = ipl_deliveries[
    ipl_deliveries["innings"].isin([1, 2])
].copy()

ipl_super_over_deliveries = ipl_deliveries[
    ipl_deliveries["innings"].isin([3, 4])
].copy()

print("REGULAR IPL DELIVERIES:", len(ipl_regular_deliveries))
print("SUPER OVER DELIVERIES:", len(ipl_super_over_deliveries))

print("\nREGULAR INNINGS DISTRIBUTION:")
display(
    ipl_regular_deliveries.groupby("innings")
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique")
    )
)

REGULAR IPL DELIVERIES: 87673
SUPER OVER DELIVERIES: 14

REGULAR INNINGS DISTRIBUTION:


,deliveries,matches
innings,,
1,45588,367
2,42085,363


In [14]:
# VALIDATE RUNS AND EXTRAS

print("RUNS COLUMN:")
display(ipl_regular_deliveries["runs"].value_counts().sort_index())

print("\nEXTRAS TYPES:")
display(
    ipl_regular_deliveries["extras_type"]
    .value_counts(dropna=False)
)

print("\nEXTRAS COLUMN SUMMARY:")
display(
    ipl_regular_deliveries[
        ["runs", "extras1_runs", "extras_2_runs"]
    ].describe()
)

RUNS COLUMN:


runs
0    33345
1    31941
2     5044
3      193
4    10963
5       13
6     6174
Name: count, dtype: int64


EXTRAS TYPES:


extras_type
NaN         82445
wide         3552
legbyes      1109
Overstep      257
byes          188
other         122
Name: count, dtype: int64


EXTRAS COLUMN SUMMARY:


,runs,extras1_runs,extras_2_runs
count,87673.000000,87673.000000,87673.000000
mean,1.409430,0.072850,0.000536
std,1.779709,0.349019,0.042044
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000
75%,2.000000,0.000000,0.000000
max,6.000000,5.000000,4.000000
